In [2]:
import numpy as np
import random
from typing import Callable, Union, Type, List
from typeguard import check_type
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score

In [55]:
class Neuron:
    def __init__(self):
        self.weights = None

    def compile(self, inp_features:int):
        self.weights = np.array([random.uniform(0, 1) for _ in range(inp_features)]).reshape(1, inp_features)  #weights shape: 1 x inp_features
        return self.weights

In [56]:
class Layer:
    def __init__(self, name:str):
        self.name = name

    def __str__(self):
        return f"Layer : {self.name}"

    def forward(self, inp:np.array):
        pass

    def backward(self, inp:np.array, prev_grad:np.array, lr:float=0.01):
        pass


class LinearLayer(Layer):
    def __init__(self, name, inp_features:int, out_features:int):
        self.name = name
        super().__init__(name)  
        self.neurons = [Neuron() for _ in range(out_features)]
        self.weights = np.array([neuron.compile(inp_features) for neuron in self.neurons]).squeeze() #weights shape : out_features x inp_features
        self.inp = None
    
    def __str__(self):
        return f"Linear Layer : {self.name}, neurons:{len(self.neurons)}"

    def forward(self, inp:np.array):#inp shape :  n x inp_features
        super()
        self.inp = inp
        # print(f"Linear layer forward : inp shape : {inp}, weights shape : {self.weights}")
        self.temp_var = inp @ self.weights.T # Z = W * X
        # print(f"Linear layer : output : {self.temp_var}")
        return self.temp_var # z shape : n x out_features
    
    def backward(self, prev_grad:np.array, lr:float=0.01): # considering pred_grad shape : 1 x out_features
        dzdw = self.inp.T # i x inp_features
        # print(f"linear layer backward : prev_grad: {prev_grad}, weights : {self.weights}")
        self.weights -= lr*(dzdw @ prev_grad).T # inp_features x out_features
        dzdx = prev_grad @ self.weights #1 x inp_features
        return dzdx.T


class Sigmoid(Layer):
    def __init__(self, name):
        self.name = name
        self.inp = None
    
    def __str__(self):
        return f"Sigmoid Layer : {self.name}"
    
    def forward(self, inp:np.array): # inp shape: n x out_features
        self.inp = np.clip(inp, -500, 500)
        self.temp_var = 1.0/(1.0+np.exp(-inp)) # n x out_features
        # print(f"sigmoid output : {self.temp_var}")
        return self.temp_var
    
    def backward(self, prev_grad:np.array, lr=0.01):#prev_grad : out_features x 1
        dsig = self.temp_var * (1-self.temp_var) #dsig shape : n x out_features
        # print(f"dsig : {dsig}")
        return dsig * prev_grad.squeeze().T


class Softmax(Layer):
    def __init__(self, name:str):
        self.name = name
        self.inp = None
    
    def __str__(self):
        return f"Softmax Layer : {self.name}"
    
    def forward(self, inp:np.array): #x shape : n x out_features
        self.inp = inp - np.max(inp, axis=1, keepdims=True)
        tempsum = np.sum(np.exp(self.inp), axis=1, keepdims=True)
        # print(f"softmax forward : tempsum : {tempsum}")
        self.temp_var = np.exp(self.inp)/tempsum
        return self.temp_var # shape : n x out_features
    
    def backward(self, prev_grad:np.array, lr = 0.01):#prev_grad shape: out_features x n
        """
        self.temp_var shape : n x out_features
        I shape : out_features x out_features
        z = I - self.temp_var shape: out_features x out_features
        self.temp_var @ z shape: 1 x out_features
        dtemp_var shape: out_features x out_features
        output_grad shape: out_features x 1
        """
        single_instance_len = len(self.temp_var[0])
        prev_grad = prev_grad[:, np.newaxis]
        output_grad = np.zeros_like(self.temp_var)
        for i in range(self.temp_var.shape[0]):
            dtemp_var = (np.diagflat(self.temp_var[i]) * np.identity(single_instance_len)) - np.outer(self.temp_var[i], self.temp_var[i])
            # print(f"dtempvar : {dtemp_var.shape}, prev grad : {prev_grad.shape}, prev grad [i] : {prev_grad[i].shape}")
            output_grad[i] = prev_grad[i] @ dtemp_var
        # print(f"softmax backward : output_grad : {output_grad}")
        return output_grad

In [57]:
def accuracy_fn(y_pred:np.array, y_true:np.array):
    # y_pred: (n, num_classes) - softmax probabilities
    # y_true: (n, num_classes) - one-hot encoded
    pred_classes = np.argmax(y_pred, axis=1)  # (n,) - class indices
    # true_classes = np.argmax(y_true, axis=1)  # (n,) - class indices
    accuracy = np.mean(pred_classes == y_true)
    return accuracy

In [58]:
class CrossEntropyLoss():
    def __init__(self):
        self.loss = None

    def calculate_loss(self, y_pred:np.array, y_true:np.array):#y_pred shape & y_true shape: n x out_features
        epsilon = 1e-15
        # print(f"y pred: {y_pred}, y_true : {y_true}")
        self.loss = np.mean(np.sum(-np.multiply(np.log(y_pred+epsilon),y_true)))# loss shape: 1 x 1
        return self.loss
    
    def backward(self, y_pred:np.array, y_true:np.array):
        grad = y_pred - y_true
        return grad#shape : out_features x n

In [59]:
class Tangrad:    
    def __init__(self, layer:list[Layer]=[], inp_features:int=None):
        self.layer = layer
        self.inp_features = inp_features
    
    def add_layer(self, layer:Union[List[Layer], Layer]):
        if check_type(layer, List[Layer]):
            for lyr in layer:
                self.layer.append(lyr)
        else:
            self.layer = layer
        
    def fit(self, inp:np.array, out:np.array, loss_fn:Type[CrossEntropyLoss], accuracy:Callable, lr:float=0.01):
        layer_inp = inp
        for i in self.layer:
            layer_inp = i.forward(layer_inp)
        epoch_loss = loss_fn.calculate_loss(layer_inp, out)
        # print(f"layer inp : {layer_inp}, out : {out}")
        epoch_acc = accuracy_score(layer_inp.argmax(axis=1), out.argmax(axis=1))
        grad = loss_fn.backward(layer_inp, out)
        for i in reversed(self.layer):
            grad = i.backward(grad, lr)
        return epoch_loss, epoch_acc

In [60]:
Ann = Tangrad(inp_features=4)

In [61]:
layer_1 = LinearLayer("Linear 1", 4, 20)
sig_layer1 = Sigmoid("Sigmoid layer")
layer_2 = LinearLayer("Linear 2", 20, 10)
sig_layer2 = Sigmoid("Sigmoid layer")
layer_3 = LinearLayer("Linear 3", 10, 3)
softmax_layer = Softmax("1")
layer_list = [layer_1, sig_layer1, layer_2, sig_layer2, layer_3, softmax_layer]
Ann.add_layer(layer_list)

In [62]:
iris  = load_iris()

In [63]:
x = iris.data[:, :]
y = iris.target
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=85)
# y_reshaped = y_train[:,np.newaxis]
# y_reshaped.shape

In [64]:
scaler = StandardScaler()
encoder = OneHotEncoder()

y_train = encoder.fit_transform(y_train[: ,np.newaxis]).toarray()
y_test = encoder.fit_transform(y_test[:,np.newaxis]).toarray()
traindata = scaler.fit_transform(x_train)
testdata = scaler.fit_transform(x_test)


In [65]:
loss = CrossEntropyLoss()
loss

In [69]:
for epoch in range(50):
    epoch_loss, epoch_acc = Ann.fit(x_train, y_train, loss, accuracy_fn, 0.02)
    print(f"Epoch : {epoch}, train loss : {epoch_loss}, train acc : {epoch_acc}")

Epoch : 0, train loss : 128.17145209803937, train acc : 0.35714285714285715
Epoch : 1, train loss : 124.92100533625612, train acc : 0.33035714285714285
Epoch : 2, train loss : 128.77115861183512, train acc : 0.35714285714285715
Epoch : 3, train loss : 128.2326336404766, train acc : 0.33035714285714285
Epoch : 4, train loss : 127.26302023426547, train acc : 0.35714285714285715
Epoch : 5, train loss : 134.19199916890258, train acc : 0.33035714285714285
Epoch : 6, train loss : 124.70822852911037, train acc : 0.35714285714285715
Epoch : 7, train loss : 127.85467255123298, train acc : 0.33035714285714285
Epoch : 8, train loss : 127.73030379211856, train acc : 0.35714285714285715
Epoch : 9, train loss : 133.7931449733978, train acc : 0.33035714285714285
Epoch : 10, train loss : 124.75065692995216, train acc : 0.35714285714285715
Epoch : 11, train loss : 128.44708231708609, train acc : 0.33035714285714285
Epoch : 12, train loss : 127.37701756610788, train acc : 0.35714285714285715
Epoch : 13,

In [139]:
s = np.random.random((1, 10))
t = s @ (np.identity(len(s.squeeze())) - s)
t

array([[-2.90895616, -5.11140815, -4.88122937, -4.36248652, -2.30521223,
        -4.89385933, -1.7624956 , -5.04745968, -3.25563507, -0.92876574]])

In [4]:
y_true = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]])
y_true

array([[0, 1, 0],
       [0, 0, 1],
       [1, 0, 0]])

In [5]:
np.argmax(y_true, axis=1)


array([1, 2, 0])